> 在线讲义：[第 22 章 · 多芯片封装与互联：计算放在哪，数据走哪条路](https://wikieden.github.io/ai-infra-from-constraints/lessons/22-package-interconnect.html) · 本书《从约束推导 AI Infra》

# 第 22 章 · 多芯片封装与互联：计算放在哪，数据走哪条路

配套讲义：仓库里 `ai-infra/notes/lessons/22-package-interconnect.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 双 die | 取最慢的一路 | `dual_die_ms(hbm, link)` | B200 8.6 / 8.6；910C 21.5 / 127 ms |
| 第二步 H2D | 字节 ÷ 带宽 | `h2d_vs_read(64 * 2**20, 32e9, 1008e9)` | 2.1 ms、67 μs、31.5 倍；$R \ge 32$ |
| 第三步 卡间 | $\alpha + V/R$ | `edge_us(V, R, alpha)` | 2.02 / 6.66 μs（H100），2.03 / 8.99 μs（A100） |
| 第三步 转折点 | $\alpha R$ | `crossover_bytes(2e-6, 450e9)` | 900,000 B ≈ 879 KiB，$M \approx 109.9$ |

B200 与昇腾 910C 的双 die 参数、RTX 4090 的 PCIe 与显存、H100 / A100 NVLink 与 2 μs 固定开销取自 AII 第 4 章 §4.5。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `dual_die_ms`**：计算都在 die 0：(本地读 ms, 跨 die 读 ms, 阶段时间 ms)。两路并行，跨 die 受远端 HBM 与链路较慢者限制。

In [ ]:
def dual_die_ms(hbm: float, link: float, gib: float = 32) -> tuple[float, float, float]:
    """计算都在 die 0：(本地读 ms, 跨 die 读 ms, 阶段时间 ms)。两路并行，跨 die 受远端 HBM 与链路较慢者限制。"""
    v = gib * 2**30
    local = v / hbm  # die 0 读自己的 32 GiB
    remote = v / min(hbm, link)  # 读 die 1 的：远端 HBM 和链路取较慢者
    return local * 1e3, remote * 1e3, max(local, remote) * 1e3

**函数 `h2d_vs_read`**：(上传 ms, 显存读一次 μs, 两者之比)。

In [ ]:
def h2d_vs_read(nbytes: float, pcie: float, hbm: float) -> tuple[float, float, float]:
    """(上传 ms, 显存读一次 μs, 两者之比)。"""
    up = nbytes / pcie
    rd = nbytes / hbm
    return up * 1e3, rd * 1e6, up / rd

**函数 `edge_us`**：点对点传输时间（μs）= 固定开销 + 数据量 / 带宽。

In [ ]:
def edge_us(nbytes: float, rate: float, alpha_s: float = 2e-6) -> float:
    """点对点传输时间（μs）= 固定开销 + 数据量 / 带宽。"""
    return (alpha_s + nbytes / rate) * 1e6

**函数 `crossover_bytes`**：固定开销与传输时间相等的数据量 V* = α·R。

In [ ]:
def crossover_bytes(alpha_s: float, rate: float) -> float:
    """固定开销与传输时间相等的数据量 V* = α·R。"""
    return alpha_s * rate

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：两颗 die 各 32 GiB 权重，计算都放在 die 0")
for name, hbm, link in (("B200", 4e12, 5e12), ("910C", 1.6e12, 270e9)):
    lo, re, st = dual_die_ms(hbm, link)
    print(f"  {name}：本地 {lo:.1f} ms，跨 die {re:.1f} ms，阶段 {st:.1f} ms（链路/本地 = {link / hbm:.2f}）")
print(f"  910C 就地计算：两侧各 {32 * 2**30 / 1.6e12 * 1e3:.1f} ms，链路只传 64 MiB {64 * 2**20 / 270e9 * 1e3:.2f} ms")

In [ ]:
print("\n第二步：RTX 4090 上传 64 MiB")
up, rd, ratio = h2d_vs_read(64 * 2**20, 32e9, 1008e9)
print(f"  H2D {up:.1f} ms，显存读 {rd:.0f} μs，相差 {ratio:.1f} 倍；复用 100 次每次分摊 {up * 1e3 / 100:.0f} μs")

In [ ]:
print("\n第三步：卡间传 [M, 4096] BF16 激活")
for M in (1, 256):
    v = 8192 * M
    print(f"  M = {M:3d}（{v // 1024} KiB）：A100 {edge_us(v, 300e9):.2f} μs，H100 {edge_us(v, 450e9):.2f} μs，H100 α 减半 {edge_us(v, 450e9, 1e-6):.2f} μs")
vs = crossover_bytes(2e-6, 450e9)
print(f"  转折点 V* = {vs:,.0f} B ≈ {vs / 1024:.0f} KiB，约 M = {vs / 8192:.1f}")

## 3. 代码里才看得到的

1. 32 GiB 是 $2^{30}$ 制，带宽是 $10^9$ 制，所以 B200 是 $34.36 \times 10^9 / 4 \times 10^{12} = 8.59$ ms。
2. B200 的 NV-HBI 按原文保守处理：即使按每方向 5 TB/s，也高于 4 TB/s 的本地 HBM，所以跨 die 由远端 HBM 决定。
3. `edge_us` 的固定开销 $\alpha$ 取 2 μs 是机内 NVLink 的教学值（原书第 7 章同一取值）；跨机时 $\alpha$ 大一个数量级。

## 4. 自己改一改

- 把 910C 的链路改成每方向 800 GB/s，计算放 die 0 还要多久？
- 用 `h2d_vs_read` 算一算 H100（PCIe 5.0 每方向 64 GB/s、3.35 TB/s）的复用下限。
- 扫 $V$ 从 1 KiB 到 64 MiB，画出 `edge_us` 在 A100 和 H100 上的曲线。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 路比出货口宽，在哪干都一样；路窄就在货边干，只运成品；
> 外地进货慢，一批货用够三十多次才不亏；
> 快递先等取件，再按重量走路；明信片换快车没用，一车货才看车速。